In [ ]:
import sempy.fabric as fabric
import time
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed

POLL_INTERVAL = 30 #segundos entre ejecución y ejecución
MAX_WAIT      = 3500 #Tiempo máximo para evitar time out
MAX_WORKERS   = 10 #número de modelos ejecutandose a la vez

# MODELOS EXCLUIDOS ────────────────────────────────────
EXCLUIDOS = {"Modelos excluidos por la razón que sea"}

def refrescar_dataset(ds_name):
    try:
        fabric.refresh_dataset(dataset=ds_name, refresh_type="Full")
    except Exception as e:
        print(f"  ❌ Error al lanzar {ds_name}: {e}")
        return ds_name, False

    elapsed = 0
    while elapsed < MAX_WAIT:
        time.sleep(POLL_INTERVAL)
        elapsed += POLL_INTERVAL
        try:
            history = fabric.list_refresh_requests(dataset=ds_name)
            if not history.empty:
                status = history.iloc[0]["Status"]
                print(f"    [{datetime.now().strftime('%H:%M:%S')}] {ds_name}: {status}")
                if status == "Completed":
                    return ds_name, True
                elif status in ["Failed", "Cancelled"]:
                    return ds_name, False
        except Exception as e:
            print(f"  ⚠️ Error leyendo estado {ds_name}: {e}")
            return ds_name, False

    print(f"  ⚠️ Timeout: {ds_name}")
    return ds_name, False

# EJECUCIÓN POR TANDAS ────────────────────────────────────────────

def ejecutar_tanda(lista, nombre_tanda):
    print(f"\n🔄 {nombre_tanda}: {len(lista)} modelos\n")
    completados, fallidos = [], []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(refrescar_dataset, ds): ds for ds in lista}
        for future in as_completed(futures):
            ds_name, ok = future.result()
            (completados if ok else fallidos).append(ds_name)
            pendientes = len(lista) - len(completados) - len(fallidos)
            print(f"  {'✅' if ok else '❌'} {ds_name} — {pendientes} pendientes")
    return completados, fallidos

# EJECUCIÓN ────────────────────────────────────────────
datasets  = fabric.list_datasets()
all_names = datasets["Dataset Name"].tolist()

excluidos_encontrados = [ds for ds in all_names if ds in EXCLUIDOS]
dataset_names         = [ds for ds in all_names if ds not in EXCLUIDOS]

if excluidos_encontrados:
    print(f"⏭️  Excluidos ({len(excluidos_encontrados)}): {', '.join(excluidos_encontrados)}\n")

inicio_global = time.time()
print(f"🚀 {len(dataset_names)} modelos en cola, máximo {MAX_WORKERS} en paralelo\n")

# Tanda 1
completados, fallidos = ejecutar_tanda(dataset_names, "Tanda 1")

# Tanda 2: reintento de modelos fallidos
completados_retry, fallidos_retry = [], []
if fallidos:
    print(f"\n⚠️  {len(fallidos)} modelos fallidos — reintentando en 60 segundos...")
    time.sleep(60)
    completados_retry, fallidos_retry = ejecutar_tanda(fallidos, "Tanda 2 — reintento")
    completados += completados_retry

# RESUMEN ──────────────────────────────────────────────
total = time.time() - inicio_global
horas, rem = divmod(int(total), 3600)
mins, segs = divmod(rem, 60)
duracion_str = f"{horas}h {mins}m {segs}s" if horas else f"{mins}m {segs}s"

print(f"\n{'='*50}")
print(f"✅ Completados ({len(completados)}): {', '.join(completados)}")
print(f"❌ Fallidos    ({len(fallidos_retry)}):    {', '.join(fallidos_retry) if fallidos_retry else 'Ninguno'}")
print(f"⏭️  Excluidos  ({len(excluidos_encontrados)}): {', '.join(excluidos_encontrados) if excluidos_encontrados else 'Ninguno'}")
print(f"⏱️ Tiempo total: {duracion_str}")
print(f"{'='*50}")

In [ ]:
# Código para ver un poco el performance de la actualizaicón programada
import sempy.fabric as fabric
import pandas as pd

WORKSPACE = "Nombre del workspace"


datasets = fabric.list_datasets(workspace=WORKSPACE)
resultados = []

for _, row in datasets.iterrows():
    ds_name = row["Dataset Name"]
    ds_id   = row["Dataset ID"]
    
    try:
        history = fabric.list_refresh_requests(dataset=ds_id, workspace=WORKSPACE)
        
        if not history.empty:
            # Coger las últimas 5 ejecuciones
            recent = history.head(5).copy()
            recent["Start Time"] = pd.to_datetime(recent["Start Time"])
            recent["End Time"]   = pd.to_datetime(recent["End Time"])
            recent["duracion_min"] = (recent["End Time"] - recent["Start Time"]).dt.total_seconds() / 60

            avg_min = recent["duracion_min"].mean()
            max_min = recent["duracion_min"].max()

            resultados.append({
                "Modelo": ds_name,
                "Avg duración (min)": round(avg_min, 1),
                "Max duración (min)": round(max_min, 1),
                "Última hora inicio": recent.iloc[0]["Start Time"].strftime("%H:%M"),
                "Último estado": recent.iloc[0]["Status"],
            })
    except Exception as e:
        resultados.append({
            "Modelo": ds_name,
            "Avg duración (min)": None,
            "Max duración (min)": None,
            "Última hora inicio": None,
            "Último estado": f"Error: {e}",
        })

# Ordenar de mayor a menor duración
df = pd.DataFrame(resultados).sort_values("Avg duración (min)", ascending=False)
print(df.to_string(index=False))

In [ ]:
#Código para lanzar una notificación al grupo de teams mediante powerapps
import requests
def enviar_alerta(fallidos, duracion_str):
    estado = "❌ Con errores" if fallidos else "✅ Satisfactorio"
    requests.post(
        "enlacepowerapps",
        json={
            "estado":   estado,
            "fallidos": ", ".join(fallidos) if fallidos else "Ninguno",
            "duracion": duracion_str
        }
    )
# Enviar la duración y si ha fallado alguno
enviar_alerta(fallidos_retry, duracion_str)